In [1]:

import os
from pathlib import Path
import duckdb

#move to the project root if needed
if Path.cwd().name == "notebooks":
    os.chdir("..")


con = duckdb.connect("data/olist.duckdb")

con.execute(Path("sql/01_load_raw.sql").read_text())

con.sql("SHOW TABLES").df()

c:\Users\Bugra\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,name
0,raw_category_translation
1,raw_customers
2,raw_order_items
3,raw_orders
4,raw_payments
5,raw_products
6,raw_reviews
7,raw_sellers


In [2]:
con.sql("SELECT COUNT(*) AS total_orders FROM raw_orders").df

<bound method DuckDBPyRelation.df of ┌──────────────┐
│ total_orders │
│    int64     │
├──────────────┤
│        99441 │
└──────────────┘
>

In [3]:
con.sql("DESCRIBE raw_orders").df()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
4,order_approved_at,TIMESTAMP,YES,None,None,None
5,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
6,order_delivered_customer_date,TIMESTAMP,YES,None,None,None
7,order_estimated_delivery_date,TIMESTAMP,YES,None,None,None


In [4]:
con.sql("""
SELECT 'orders' AS table_name, COUNT(*) AS n_rows FROM raw_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM raw_order_items
UNION ALL SELECT 'customers', COUNT(*) FROM raw_customers
UNION ALL SELECT 'payments', COUNT(*) FROM raw_payments
UNION ALL SELECT 'reviews', COUNT(*) FROM raw_reviews
UNION ALL SELECT 'products', COUNT(*) FROM raw_products
UNION ALL SELECT 'sellers', COUNT(*) FROM raw_sellers;
""").df()

,table_name,n_rows
0,orders,99441
1,order_items,112650
2,customers,99441
3,payments,103886
4,reviews,99224
5,products,32951
6,sellers,3095


In [ ]:
#checked for duplicate orders

con.sql("""
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT order_id) AS n_unique_orders
FROM raw_orders; 
""").df()

,n_rows,n_unique_orders
0,99441,99441


In [6]:
#checked for orders with multiple reviews

con.sql("""
SELECT COUNT(*) AS orders_with_several_reviews
FROM (
    SELECT order_id
    FROM raw_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1);
""").df()

,orders_with_several_reviews
0,547


In [ ]:
#keep the latest review for each order, with this code, we check the number for each delivery status

con.sql("""
SELECT order_status,
       COUNT (*) AS n_orders
FROM raw_orders
GROUP BY  order_status
ORDER BY n_orders DESC;
""").df()

,order_status,n_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [8]:
con.sql("""
SELECT COUNT(*) AS non_delivered_orders
FROM raw_orders
WHERE order_status <> 'delivered';
""").df()

,non_delivered_orders
0,2963


In [9]:
con.sql("""
SELECT COUNT (*) AS delivered_without_date
FROM raw_orders
WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NULL;
""").df()

,delivered_without_date
0,8


In [ ]:
#Look at the orders by date. We determined that we should include from January 2017 through August 2018

con.sql("""
SELECT date_trunc('month', order_purchase_timestamp) AS month,
       COUNT(*) AS n_orders
FROM raw_orders
GROUP BY month
ORDER BY month;
""").df()

,month,n_orders
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


In [ ]:
#number of products without categories. We should replace them with unknown

con.sql("""
SELECT COUNT(*) AS products_without_categories
FROM raw_products
WHERE product_category_name IS NULL
""").df()

,products_without_categories
0,610


In [ ]:
#we should use customer_unique_id when analyzing distinct customers

con.sql("""
SELECT COUNT(DISTINCT customer_id) AS customer_ids,
       COUNT(DISTINCT customer_unique_id) AS real_customers
FROM raw_customers;
""").df()

,customer_ids,real_customers
0,99441,96096
